# DDPM：去噪扩散概率模型 on CIFAR-10

这个 Notebook 从零实现 DDPM（Denoising Diffusion Probabilistic Models，Ho et al. 2020）并在 CIFAR-10 上训练图像生成模型。

内容包括：
- 线性 Beta Schedule 与前向加噪过程
- 不同时间步下的加噪效果可视化
- U-Net 去噪网络（含时间步嵌入、ResBlock、Self-Attention、Skip Connection）
- 训练目标：预测噪声（简化目标）
- DDPM 逆向采样（Algorithm 2）
- 生成图像展示

## 1. 环境准备

```bash
pip install torch torchvision matplotlib
```

In [ ]:
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    data_root: str    = './data'
    image_size: int   = 32       # CIFAR-10 原始尺寸，生成模型不需要放大
    channels: int     = 3
    batch_size: int   = 128
    num_workers: int  = 2
    lr: float         = 2e-4
    epochs: int       = 5
    # 扩散过程参数
    T: int            = 1000     # 总时间步数
    beta_start: float = 1e-4
    beta_end: float   = 0.02
    # U-Net 参数
    base_channels: int = 64      # 第一层通道数，后续翻倍


cfg = Config()
cfg

## 2. 加载 CIFAR-10

生成模型使用原始 32×32 分辨率，归一化到 `[-1, 1]`（mean=0.5, std=0.5）与加噪噪声的标准高斯分布对齐。

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.CIFAR10(root=cfg.data_root, train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root=cfg.data_root, train=False, download=True, transform=transform)
classes       = train_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available())

images, _ = next(iter(train_loader))
print('batch shape:', images.shape)

In [ ]:
def denorm(x):
    return (x.clamp(-1, 1) + 1) / 2


fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.flatten(), range(8)):
    img, lbl = train_dataset[idx]
    ax.imshow(denorm(img).permute(1, 2, 0))
    ax.set_title(classes[lbl])
    ax.axis('off')
plt.suptitle('CIFAR-10 samples (32×32)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. 扩散过程（前向加噪）

前向过程逐步向图像添加高斯噪声：

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\, \beta_t \mathbf{I})$$

**Closed-form 加噪**（无需逐步采样）：

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\, (1-\bar{\alpha}_t) \mathbf{I})$$

其中 $\bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s)$。

In [ ]:
class DiffusionScheduler:
    def __init__(self, cfg):
        self.T = cfg.T
        # 线性 beta schedule
        betas              = torch.linspace(cfg.beta_start, cfg.beta_end, cfg.T)
        alphas             = 1.0 - betas
        alphas_cumprod     = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        # 预计算常用系数，避免每步重复计算
        self.sqrt_alphas_cumprod      = alphas_cumprod.sqrt()
        self.sqrt_one_minus_alphas_cp = (1 - alphas_cumprod).sqrt()
        self.betas                    = betas
        self.alphas                   = alphas
        self.alphas_cumprod           = alphas_cumprod
        # 逆向过程的后验方差
        self.posterior_variance = betas * (1 - alphas_cumprod_prev) / (1 - alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        # 直接从 x0 采样任意时间步 t 的加噪结果（closed-form）
        if noise is None:
            noise = torch.randn_like(x0)
        s1 = self._extract(self.sqrt_alphas_cumprod, t, x0)
        s2 = self._extract(self.sqrt_one_minus_alphas_cp, t, x0)
        return s1 * x0 + s2 * noise, noise

    @torch.no_grad()
    def p_sample(self, model, x, t_scalar):
        # 逆向过程一步：x_t → x_{t-1}
        t_batch = torch.full((x.size(0),), t_scalar, device=x.device, dtype=torch.long)
        eps_pred = model(x, t_batch)

        beta    = self._extract(self.betas,                    t_batch, x)
        alpha   = self._extract(self.alphas,                   t_batch, x)
        s_omac  = self._extract(self.sqrt_one_minus_alphas_cp, t_batch, x)

        # 预测 x_0 并计算 x_{t-1} 均值
        mean = (1 / alpha.sqrt()) * (x - beta / s_omac * eps_pred)

        if t_scalar == 0:
            return mean
        var   = self._extract(self.posterior_variance, t_batch, x)
        noise = torch.randn_like(x)
        return mean + var.sqrt() * noise

    @staticmethod
    def _extract(coef, t, x):
        # 把 1D 系数 coef[t] 广播到与 x 相同形状
        val = coef[t.cpu()].to(x.device)
        return val.view(-1, *([1] * (x.ndim - 1)))


scheduler = DiffusionScheduler(cfg)

In [ ]:
# 可视化同一张图在不同时间步的加噪效果
x0, _ = train_dataset[0]
x0    = x0.unsqueeze(0)

timesteps = [0, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(16, 3))

for ax, t in zip(axes, timesteps):
    t_tensor = torch.tensor([t])
    xt, _    = scheduler.q_sample(x0, t_tensor)
    ax.imshow(denorm(xt[0]).permute(1, 2, 0))
    ax.set_title(f't={t}')
    ax.axis('off')

plt.suptitle('前向加噪过程（t=0 原图 → t=999 纯噪声）', fontsize=13)
plt.tight_layout()
plt.show()

## 4. U-Net 去噪网络

U-Net 接受加噪图像 $x_t$ 和时间步 $t$，输出预测噪声 $\hat{\epsilon}$。

- **时间步嵌入**：正弦编码 + 两层 MLP，加入每个 ResBlock
- **ResBlock**：两个 Conv+GroupNorm+SiLU，时间步 embedding 用加法注入
- **Self-Attention**：在 8×8 分辨率处加入，捕获全局结构
- **Skip Connection**：下采样路径的特征图 concat 到上采样路径

In [ ]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half  = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args  = t.float()[:, None] * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1    = nn.Sequential(nn.GroupNorm(8, in_ch),  nn.SiLU(), nn.Conv2d(in_ch,  out_ch, 3, padding=1))
        self.conv2    = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        # 当 in_ch != out_ch 时需要 1x1 卷积对齐 shortcut
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        # 时间步嵌入通过加法注入到空间特征
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.shortcut(x)


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.attn = nn.MultiheadAttention(channels, num_heads=4, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        # 展平空间维度作为序列
        h = self.norm(x).view(B, C, H * W).transpose(1, 2)
        h, _ = self.attn(h, h, h)
        return x + h.transpose(1, 2).view(B, C, H, W)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=256):
        super().__init__()
        C = base_channels

        # 时间步嵌入：正弦编码 → MLP
        self.time_embed = nn.Sequential(
            SinusoidalEmbedding(C),
            nn.Linear(C, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Encoder（下采样路径）
        self.enc0 = nn.Conv2d(in_channels, C, 3, padding=1)          # 32x32
        self.enc1 = ResBlock(C,     C,     time_dim)                  # 32x32
        self.down1 = nn.Conv2d(C,   C * 2, 4, stride=2, padding=1)   # 16x16
        self.enc2 = ResBlock(C * 2, C * 2, time_dim)                  # 16x16
        self.down2 = nn.Conv2d(C*2, C * 4, 4, stride=2, padding=1)   # 8x8
        self.enc3 = ResBlock(C * 4, C * 4, time_dim)                  # 8x8

        # Bottleneck（含 Self-Attention）
        self.mid1 = ResBlock(C * 4, C * 4, time_dim)
        self.attn = SelfAttention(C * 4)
        self.mid2 = ResBlock(C * 4, C * 4, time_dim)

        # Decoder（上采样路径，concat skip connection 后通道翻倍再压缩）
        self.up2   = nn.ConvTranspose2d(C * 4, C * 2, 4, stride=2, padding=1)  # 16x16
        self.dec2  = ResBlock(C * 4, C * 2, time_dim)   # concat enc2: C*2+C*2=C*4 in
        self.up1   = nn.ConvTranspose2d(C * 2, C,     4, stride=2, padding=1)  # 32x32
        self.dec1  = ResBlock(C * 2, C,     time_dim)   # concat enc1: C+C=C*2 in

        # 输出层
        self.out = nn.Sequential(
            nn.GroupNorm(8, C),
            nn.SiLU(),
            nn.Conv2d(C, in_channels, 3, padding=1),
        )

    def forward(self, x, t):
        t_emb = self.time_embed(t)

        # 下采样路径
        h0 = self.enc0(x)
        h1 = self.enc1(h0, t_emb)    # skip1
        h  = self.down1(h1)
        h2 = self.enc2(h, t_emb)     # skip2
        h  = self.down2(h2)
        h  = self.enc3(h, t_emb)

        # Bottleneck
        h  = self.mid1(h, t_emb)
        h  = self.attn(h)
        h  = self.mid2(h, t_emb)

        # 上采样路径（concat skip）
        h  = self.up2(h)
        h  = self.dec2(torch.cat([h, h2], dim=1), t_emb)
        h  = self.up1(h)
        h  = self.dec1(torch.cat([h, h1], dim=1), t_emb)

        return self.out(h)


unet = UNet(in_channels=cfg.channels, base_channels=cfg.base_channels).to(device)
unet

In [ ]:
@torch.no_grad()
def inspect_unet_shapes(model, cfg):
    x = torch.randn(2, cfg.channels, cfg.image_size, cfg.image_size)
    t = torch.randint(0, cfg.T, (2,))
    print(f'input  : {tuple(x.shape)}')
    print(f't      : {tuple(t.shape)}')
    out = model(x, t)
    print(f'output : {tuple(out.shape)}  (预测噪声，与输入同形状)')


inspect_unet_shapes(unet.cpu(), cfg)
unet = unet.to(device)

## 5. 关键机制解读

### 前向过程 vs 逆向过程
| | 前向 q | 逆向 p |
|---|---|---|
| 方向 | x_0 → x_T（加噪） | x_T → x_0（去噪） |
| 是否有参数 | 无，固定马尔可夫链 | 有，U-Net 预测噪声 |
| 关键公式 | closed-form 任意 t 采样 | 逐步去噪，共 T 步 |

### 简化训练目标
- 原始 DDPM 目标等价于：$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}[\|\epsilon - \epsilon_\theta(x_t, t)\|^2]$
- 随机采样 t，从 x_0 一步加噪得到 x_t，让模型预测所加的噪声 $\epsilon$。

### 时间步嵌入
- 正弦位置编码（与 Transformer 相同原理），让 U-Net 知道当前处于哪个噪声水平。
- 通过加法注入每个 ResBlock，使去噪强度随 t 自适应调整。

### Skip Connection
- 下采样路径的低层细节通过 skip concat 传到上采样路径，帮助恢复高频纹理。

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f'Trainable parameters: {count_parameters(unet):,}')

## 6. 训练函数

In [ ]:
optimizer = optim.AdamW(unet.parameters(), lr=cfg.lr)


def train_one_epoch(model, scheduler, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total      = 0
    for x0, _ in loader:
        x0    = x0.to(device)
        # 对每个样本随机采样时间步
        t     = torch.randint(0, scheduler.T, (x0.size(0),), device=device)
        xt, noise = scheduler.q_sample(x0, t)
        xt    = xt.to(device)
        noise = noise.to(device)
        # 预测噪声，计算 MSE
        pred  = model(xt, t)
        loss  = F.mse_loss(pred, noise)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x0.size(0)
        total      += x0.size(0)
    return total_loss / total

## 7. 训练主循环

In [ ]:
history = {'loss': []}

for epoch in range(cfg.epochs):
    loss = train_one_epoch(unet, scheduler, train_loader, optimizer, device)
    history['loss'].append(loss)
    print(f'Epoch [{epoch + 1}/{cfg.epochs}]  loss={loss:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history['loss']) + 1), history['loss'])
plt.title('Training Loss (MSE)')
plt.xlabel('Epoch')
plt.tight_layout()
plt.show()

## 8. 采样生成图像（DDPM Algorithm 2）

In [ ]:
@torch.no_grad()
def sample(model, scheduler, n_samples, cfg, device):
    model.eval()
    # 从纯高斯噪声开始
    x = torch.randn(n_samples, cfg.channels, cfg.image_size, cfg.image_size, device=device)
    # 逐步去噪，共 T 步
    for t in reversed(range(scheduler.T)):
        x = scheduler.p_sample(model, x, t)
    return x.clamp(-1, 1)


generated = sample(unet, scheduler, n_samples=16, cfg=cfg, device=device)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img in zip(axes.flatten(), generated):
    ax.imshow(denorm(img.cpu()).permute(1, 2, 0))
    ax.axis('off')
plt.suptitle('DDPM 生成图像（训练 {} epochs）'.format(cfg.epochs), fontsize=14)
plt.tight_layout()
plt.show()